In [1]:
# DM4ML -Assignment - Ingestion - File

# ============================================================
# Retailrocket Ingestion Notebook / Script
# Purpose:
#   - Download Retailrocket dataset via kagglehub
#   - Copy raw files into partitioned local data lake folders
#   - Log success/failure events
#   - Create a manifest for audit/lineage
#   - Build bronze parquet datasets for downstream use
# ============================================================

# -----------------------------
# 0) Optional: install packages
# Uncomment if needed in Jupyter
# -----------------------------
# import sys
# !{sys.executable} -m pip install kagglehub pandas pyarrow

import json
import shutil
import time
from pathlib import Path
from datetime import datetime, timezone

import pandas as pd
import kagglehub

# =============================
# 1) CONFIG
# =============================
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / "data"
SOURCE_NAME = "retailrocket"
DATASET_ID = "retailrocket/ecommerce-dataset"

UTC_NOW = datetime.now(timezone.utc)
LOAD_DATE = UTC_NOW.strftime("%Y-%m-%d")
LOAD_HOUR = UTC_NOW.strftime("%H")
BATCH_ID = UTC_NOW.strftime("%Y%m%dT%H%M%SZ")
INGESTION_TS = UTC_NOW.isoformat()

RAW_DIR = PROJECT_ROOT / "data" / "raw" / SOURCE_NAME / f"load_date={LOAD_DATE}" / f"load_hour={LOAD_HOUR}"
BRONZE_DIR = PROJECT_ROOT / "data" / "bronze" / SOURCE_NAME
LOG_DIR = PROJECT_ROOT / "logs"
METADATA_DIR = PROJECT_ROOT / "metadata"

RAW_DIR.mkdir(parents=True, exist_ok=True)
BRONZE_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / f"{SOURCE_NAME}_ingestion_log.jsonl"
MANIFEST_FILE = METADATA_DIR / f"{SOURCE_NAME}_manifest_{BATCH_ID}.json"

EXPECTED_FILES = [
    "events.csv",
    "category_tree.csv",
    "item_properties_part1.csv",
    "item_properties_part2.csv",
]

# =============================
# 2) LOGGING
# =============================
def log_event(stage, status, message, extra=None):
    record = {
        "event_ts": datetime.now(timezone.utc).isoformat(),
        "source": SOURCE_NAME,
        "stage": stage,
        "status": status,
        "message": message,
        "batch_id": BATCH_ID,
    }
    if extra:
        record.update(extra)

    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")

# =============================
# 3) RETRY WRAPPER
# =============================
def retry_download(dataset_id, max_attempts=3, wait_seconds=5):
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            log_event(
                stage="download",
                status="started",
                message=f"Download attempt {attempt} started",
                extra={"dataset_id": dataset_id},
            )
            path = kagglehub.dataset_download(dataset_id)
            log_event(
                stage="download",
                status="success",
                message=f"Download attempt {attempt} succeeded",
                extra={"dataset_path": str(path)},
            )
            return Path(path)
        except Exception as e:
            last_error = str(e)
            log_event(
                stage="download",
                status="failed",
                message=f"Download attempt {attempt} failed",
                extra={"error": last_error},
            )
            if attempt < max_attempts:
                time.sleep(wait_seconds)

    raise RuntimeError(f"Download failed after {max_attempts} attempts. Last error: {last_error}")

# =============================
# 4) DOWNLOAD DATASET
# =============================
dataset_path = retry_download(DATASET_ID, max_attempts=3, wait_seconds=5)
print("Downloaded dataset to:", dataset_path)

# =============================
# 5) DISCOVER FILES
# =============================
all_files = [p for p in dataset_path.rglob("*") if p.is_file()]
file_lookup = {p.name: p for p in all_files}

missing_files = [f for f in EXPECTED_FILES if f not in file_lookup]
if missing_files:
    log_event(
        stage="file_validation",
        status="failed",
        message="Expected files missing",
        extra={"missing_files": missing_files},
    )
    raise FileNotFoundError(f"Missing expected files: {missing_files}")

log_event(
    stage="file_validation",
    status="success",
    message="All expected files found",
    extra={"found_files": EXPECTED_FILES},
)

# =============================
# 6) COPY RAW FILES TO DATA LAKE
# =============================
manifest = {
    "source": SOURCE_NAME,
    "dataset_id": DATASET_ID,
    "batch_id": BATCH_ID,
    "ingestion_ts": INGESTION_TS,
    "raw_output_dir": str(RAW_DIR),
    "files": [],
}

for file_name in EXPECTED_FILES:
    src = file_lookup[file_name]
    dest = RAW_DIR / file_name
    shutil.copy2(src, dest)

    file_record = {
        "file_name": file_name,
        "source_path": str(src),
        "raw_path": str(dest),
        "copied_at": datetime.now(timezone.utc).isoformat(),
        "file_size_bytes": dest.stat().st_size,
    }
    manifest["files"].append(file_record)

log_event(
    stage="raw_storage",
    status="success",
    message="Raw files copied successfully",
    extra={"raw_dir": str(RAW_DIR)},
)

with open(MANIFEST_FILE, "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

log_event(
    stage="manifest",
    status="success",
    message="Manifest created",
    extra={"manifest_file": str(MANIFEST_FILE)},
)

print("Raw files copied to:", RAW_DIR)
print("Manifest saved to:", MANIFEST_FILE)

# =============================
# 7) LOAD RAW CSVs
# =============================
events_df = pd.read_csv(RAW_DIR / "events.csv")
category_tree_df = pd.read_csv(RAW_DIR / "category_tree.csv")
item_properties_1_df = pd.read_csv(RAW_DIR / "item_properties_part1.csv")
item_properties_2_df = pd.read_csv(RAW_DIR / "item_properties_part2.csv")

item_properties_df = pd.concat(
    [item_properties_1_df, item_properties_2_df],
    ignore_index=True,
)

log_event(
    stage="raw_read",
    status="success",
    message="Raw CSV files loaded into pandas",
    extra={
        "events_rows": int(len(events_df)),
        "category_tree_rows": int(len(category_tree_df)),
        "item_properties_rows": int(len(item_properties_df)),
    },
)

# =============================
# 8) LIGHT BRONZE STANDARDIZATION
# =============================
def add_ingestion_metadata(df, source_file):
    out = df.copy()
    out["source_system"] = SOURCE_NAME
    out["source_file"] = source_file
    out["batch_id"] = BATCH_ID
    out["ingestion_ts"] = INGESTION_TS
    return out

events_bronze = add_ingestion_metadata(events_df, "events.csv")
category_tree_bronze = add_ingestion_metadata(category_tree_df, "category_tree.csv")
item_properties_bronze = add_ingestion_metadata(item_properties_df, "item_properties_combined.csv")

if "timestamp" in events_bronze.columns:
    events_bronze["event_ts"] = pd.to_datetime(events_bronze["timestamp"], unit="ms", errors="coerce")

if "timestamp" in item_properties_bronze.columns:
    item_properties_bronze["property_ts"] = pd.to_datetime(
        item_properties_bronze["timestamp"], unit="ms", errors="coerce"
    )

# =============================
# 9) WRITE BRONZE DATASETS
# =============================
events_out = BRONZE_DIR / "events.parquet"
category_tree_out = BRONZE_DIR / "category_tree.parquet"
item_properties_out = BRONZE_DIR / "item_properties.parquet"
summary_out = BRONZE_DIR / "ingestion_summary.csv"

events_bronze.to_parquet(events_out, index=False)
category_tree_bronze.to_parquet(category_tree_out, index=False)
item_properties_bronze.to_parquet(item_properties_out, index=False)

summary_df = pd.DataFrame(
    [
        {
            "dataset": "events",
            "rows": len(events_bronze),
            "columns": len(events_bronze.columns),
            "output_path": str(events_out),
        },
        {
            "dataset": "category_tree",
            "rows": len(category_tree_bronze),
            "columns": len(category_tree_bronze.columns),
            "output_path": str(category_tree_out),
        },
        {
            "dataset": "item_properties",
            "rows": len(item_properties_bronze),
            "columns": len(item_properties_bronze.columns),
            "output_path": str(item_properties_out),
        },
    ]
)

summary_df.to_csv(summary_out, index=False)

log_event(
    stage="bronze_write",
    status="success",
    message="Bronze datasets written successfully",
    extra={
        "events_out": str(events_out),
        "category_tree_out": str(category_tree_out),
        "item_properties_out": str(item_properties_out),
        "summary_out": str(summary_out),
    },
)

# =============================
# 10) PREVIEW
# =============================
print("\n=== Ingestion Summary ===")
print(summary_df)

print("\n=== Events Sample ===")
print(events_bronze.head())

print("\n=== Category Tree Sample ===")
print(category_tree_bronze.head())

print("\n=== Item Properties Sample ===")
print(item_properties_bronze.head())

log_event(
    stage="pipeline",
    status="success",
    message="Retailrocket ingestion pipeline completed successfully",
)


c:\Users\barath\AppData\Local\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Downloaded dataset to: C:\Users\barath\.cache\kagglehub\datasets\retailrocket\ecommerce-dataset\versions\2


Raw files copied to: C:\Users\barath\recomart-pipeline\data\raw\retailrocket\load_date=2026-04-29\load_hour=07
Manifest saved to: C:\Users\barath\recomart-pipeline\metadata\retailrocket_manifest_20260429T075249Z.json



=== Ingestion Summary ===
           dataset      rows  columns  \
0           events   2756101       10   
1    category_tree      1669        6   
2  item_properties  20275902        9   

                                         output_path  
0  C:\Users\barath\recomart-pipeline\data\bronze\...  
1  C:\Users\barath\recomart-pipeline\data\bronze\...  
2  C:\Users\barath\recomart-pipeline\data\bronze\...  

=== Events Sample ===
       timestamp  visitorid event  itemid  transactionid source_system  \
0  1433221332117     257597  view  355908            NaN  retailrocket   
1  1433224214164     992329  view  248676            NaN  retailrocket   
2  1433221999827     111016  view  318965            NaN  retailrocket   
3  1433221955914     483717  view  253185            NaN  retailrocket   
4  1433221337106     951259  view  367447            NaN  retailrocket   

  source_file          batch_id                      ingestion_ts  \
0  events.csv  20260429T075249Z  2026-04-29T07:52:4